This is the code for [homework week 1](https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw1)

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


In [2]:
from minsearch import Index  #创建索引

index = Index(
    text_fields=["content"],  
    keyword_fields=["filename"]   
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5  
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [4]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client=OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://apihub.agnes-ai.com/v1")

def build_context(search_results): 
    lines = []

    for doc in search_results:
        lines.append(doc["content"])
        lines.append(doc["filename"])
        lines.append("")

    return "\n".join(lines).strip()

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
""" 

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

def llmusage(instructions, user_prompt, model='agnes-2.0-flash'):
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.usage

llmusage(INSTRUCTIONS, prompt)


CompletionUsage(completion_tokens=255, prompt_tokens=7788, total_tokens=8043, completion_tokens_details=None, prompt_tokens_details=None)

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [25]:
from minsearch import Index

chunks_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunks_index.fit(chunks)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5  
)

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client=OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://apihub.agnes-ai.com/v1")

def build_context(search_results): 
    lines = []

    for doc in search_results:
        lines.append(doc["content"])
        lines.append(doc["filename"])
        lines.append("")

    return "\n".join(lines).strip()

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
""" 

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

def llmusage(instructions, user_prompt, model='agnes-2.0-flash'):
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.usage

llmusage(INSTRUCTIONS, prompt)


CompletionUsage(completion_tokens=225, prompt_tokens=7788, total_tokens=8013, completion_tokens_details=None, prompt_tokens_details=None)

In [30]:
class TeachingAssistantAgent:
    def __init__(self, index, instructions, model='agnes-2.0-flash'):
        self.index = index
        self.instructions = instructions
        self.model = model
        self.run_count = 0  # track number of runs

    def search(self, query, top_k=5):
        """Search the index and return top_k chunks"""
        return self.index.search(query, num_results=top_k)

    def run(self, question):
        """Agent loop: perform multiple searches before answering"""
        # Increment run counter
        self.run_count += 1

        # 1. create a few keyword variations
        keywords = [
            question,       # full question
            "limitations",  # general keyword
            "example",      # another angle
            "summary"
        ]

        # 2. collect results from multiple searches
        collected_chunks = []
        for kw in keywords:
            results = self.search(kw, top_k=5)
            collected_chunks.extend(results)

        # 3. deduplicate by (filename, start)
        seen = set()
        unique_chunks = []
        for chunk in collected_chunks:
            key = (chunk.get("filename"), chunk.get("start"))
            if key not in seen:
                seen.add(key)
                unique_chunks.append(chunk)

        # 4. build prompt with up to 10 chunks
        prompt = build_prompt(question, unique_chunks[:10])

        # 5. call LLM
        messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = openai_client.chat.completions.create(
            model=self.model,
            messages=messages
        )

        return response.choices[0].message.content

# -------------------------
# Example usage
agent = TeachingAssistantAgent(index, INSTRUCTIONS)
question = "How does the agentic loop work, and how is it different from plain RAG?"
answer = agent.run(question)
print(f"Run #{agent.run_count} answer:\n{answer}")

Run #1 answer:
Based on the provided context, here is how the agentic loop works and how it differs from plain RAG:

**How the Agentic Loop Works**
The agentic loop is a `while True` loop (or similar iterative pattern) where the LLM is in charge of the decision-making process. It functions as follows:
1.  The LLM receives a prompt and a list of available tools.
2.  The LLM decides whether to answer the user directly or call a specific tool.
3.  If a tool is called, the system executes the function, retrieves the result, and sends it back to the LLM as part of the conversation history.
4.  The LLM evaluates the result. If more information is needed (e.g., due to a typo or insufficient data), it may call another tool or retry with modified parameters.
5.  This cycle repeats until the LLM determines it has enough information to provide a final answer, at which point the loop stops.

**Difference from Plain RAG**
The primary difference is **who controls the flow**:

*   **Plain RAG (Fixed 